<a href="https://colab.research.google.com/github/JoseGG8/Equipo3_-CodeFactory/blob/main/Laboratorio/Laboratorio_Hashes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Punto #1**


In [2]:
import hashlib
def hash(key):
  hash_resultado = hashlib.sha256(key.encode('utf-8')).hexdigest()
  return hash_resultado

In [4]:
import itertools

# --- CONFIGURACIÓN DEL OBJETIVO ---
# Definimos la secuencia original que queremos "descifrar"
SECUENCIA = str(input("ingresa una secuencia de 10 digitos"))
# Calculamos el hash de referencia (Nota: en Python, hash() cambia por sesión por seguridad)
hash_resultado = hash(SECUENCIA)
print(f"hash a encontrar: {hash_resultado}")

# --- PARÁMETROS DE BÚSQUEDA ---
digitos = '0123456789' # Espacio de caracteres (decimal)
longitud = 10         # Longitud exacta de la cadena a buscar

# Generador de todas las combinaciones posibles con repetición (Producto cartesiano)
# Esto genera un espacio de búsqueda de 10^10 (10,000,000,000 de posibilidades)
combinaciones = itertools.product(digitos, repeat=longitud)

# --- BÚSQUEDA POR FUERZA BRUTA ---
# islice limita la búsqueda para evitar bucles infinitos si no se encuentra el valor
if len(SECUENCIA)==10:
  for i, combinacion in enumerate(itertools.islice(combinaciones, 100000000000)):
      # Convertimos la tupla generada por product en un string (ej: ('0','1'...) -> "01...")
      numero = ''.join(combinacion)

      # Comparamos el hash de la combinación actual con el objetivo
      if hash(numero) == hash_resultado:
          print(f"{numero} genera el hash {hash(numero)}")
          # Al encontrar la coincidencia, detenemos la ejecución
          break

ingresa una secuencia de 10 digitos0000000000
hash a encontrar: 2211789646767583795
0000000000 genera el hash 2211789646767583795


**Punto #2**

In [32]:
##ARBOL DE MERKLE
import hashlib

# Clase para representar cada nodo del árbol (hojas, nodos intermedios y raíz)
class Nodo:
    def __init__(self, key):
        self.key = key      # Almacena el valor hash del nodo
        self.pair = None    # Referencia al nodo hermano (complemento para el siguiente hash)

class MerkleArbol:
    def __init__(self, initial_keys):
        self.root = None
        self.initial_keys = initial_keys    # Lista de datos/transacciones originales
        self.leafs_nodos = []               # Almacenará los nodos hoja ya hasheados

    # Genera un hash SHA-256 a partir de una cadena de texto
    def hash(self, key):
        return hashlib.sha256(key.encode("utf-8")).hexdigest()

    # Concatena dos hashes y devuelve el hash del resultado (hijo izquierdo + hijo derecho)
    def hash_pairs(self, hash1, hash2):
        return self.hash(hash1 + hash2)

    # Convierte las transacciones iniciales en los nodos hoja del árbol (primer nivel de hashing)
    def hash_initialKeys(self):
        for key in self.initial_keys:
            self.leafs_nodos.append(Nodo(self.hash(key)))

    # Proceso iterativo para construir el árbol desde las hojas hasta la raíz
    def build_tree(self):
        nodes = self.leafs_nodos

        # Se repite el proceso hasta que solo quede un nodo (la raíz)
        while len(nodes) > 1:
            next_level = []

            # Si el número de nodos es impar, se duplica el último para balancear el árbol
            if len(nodes) % 2 != 0:
                nodes.append(Nodo(nodes[-1].key))

            # Agrupa los nodos de dos en dos para generar el nivel superior
            for i in range(0, len(nodes), 2):
                left = nodes[i]
                right = nodes[i+1]

                # Establece la relación de hermandad entre los nodos
                left.pair = right
                right.pair = left

                # Calcula el hash del padre a partir de los dos hijos
                parent_hash = self.hash_pairs(left.key, right.key)
                next_level.append(Nodo(parent_hash))

            # El nivel recién generado se convierte en la base para la siguiente iteración
            nodes = next_level

        # Asigna y devuelve el Merkle Root final
        self.root = nodes[0] if nodes else None
        return self.root


In [31]:
"""
Script para la recuperación del orden de transacciones mediante Merkle Root.
"""

from itertools import permutations
import random

# --- CONFIGURACIÓN DE REFERENCIA ---
# Definimos el estado original "secreto"
TRANSACCIONES_REFERENCIA = ["hola2", "hola1", "hola4", "hola3"]
merkle_tree = MerkleArbol(TRANSACCIONES_REFERENCIA)
merkle_tree.hash_initialKeys()
root_referencia = merkle_tree.build_tree()

print(f"ROOT DE REFERENCIA: {root_referencia.key}\n")

# Mostramos las transacciones al usuario en orden aleatorio para no dar pistas
print(f"Transacciones conocidas (orden aleatorio):")
random.shuffle(TRANSACCIONES_REFERENCIA)
for i, transaccion in enumerate(TRANSACCIONES_REFERENCIA):
    print(f"{i+1}. {transaccion}")

# --- ENTRADA DE DATOS ---
TRANSACCIONES_input = input("\nIngrese las transacciones separadas por coma para iniciar búsqueda: \n")
TRANSACCIONES = [t.strip() for t in TRANSACCIONES_input.split(",")]

# --- MOTOR DE BÚSQUEDA ---
# Generamos el espacio de búsqueda (permutaciones)
permutaciones = [list(p) for p in permutations(TRANSACCIONES)]

for permutacion in permutaciones:
    # Reconstrucción tentativa del árbol
    merkle_tree_prueba = MerkleArbol(permutacion)
    merkle_tree_prueba.hash_initialKeys()
    root_a_comparar = merkle_tree_prueba.build_tree()

    # Verificación de integridad contra la referencia
    if root_referencia.key == root_a_comparar.key:
        print(f"\n[ÉXITO] Coincidencia encontrada.")
        print(f"Hash Root validado: {root_a_comparar.key}")
        break

# --- SALIDA DE RESULTADOS ---
print("\nORDEN DETERMINADO DE LAS TRANSACCIONES:")
for i, transaccion in enumerate(permutacion):
    print(f"{i+1}. {transaccion}")

el root de referencia es 210d3613b24c917d2a7ceb359ad443565eb4fbff65d1c1638235db33206b3e76 

las transacciones posibles son: 
1. hola2 
2. hola4 
3. hola1 
4. hola3 
Ingrese las transacciones separadas por coma (ej: hola1,hola2,hola3): hola1,hola2,hola3,hola4
la permutacion ['hola2', 'hola1', 'hola4', 'hola3'] de las transacciones en este orden genera el root 210d3613b24c917d2a7ceb359ad443565eb4fbff65d1c1638235db33206b3e76 

Orden de las transacciones:
1. hola2 
2. hola1 
3. hola4 
4. hola3 
